# 再利用可能な埋め込み（Reusable Embeddings）

**学習目標**
1. 事前学習済みの Kaggle Hub テキストモジュールを使用して文ベクトルを生成する方法を学ぶ
1. 事前学習済みの Kaggle Hub モジュールを Keras モデルに組み込む方法を学ぶ


## はじめに


このノートブックでは、タイトルデータセットにある記事タイトルの推定されるソース（GitHub、TechCrunch、または The New York Times）を識別するテキストモデルを実装します。

まず、最初のレイヤとして Kaggle Hub の事前学習済みモジュールを持つ Keras モデルに入力できるように、テキストとラベルを読み込んで前処理します。この最初のレイヤのおかげで、モデルに渡す前にテキストをトークン化したり整数化したりする必要がありません。事前学習済みのレイヤがそれを処理してくれるため、未加工の生テキストを直接受け取ることができます。ただし、3 つのクラスそれぞれを 3 次元の基底ベクトルにワンホットエンコーディングする必要があります。

次に、1 つ以上の事前学習済み Kaggle Hub レイヤから始まるシンプルな DNN モデルを構築、訓練、比較します。

In [ ]:
import os
import warnings

import pandas as pd
from google.cloud import bigquery

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

%load_ext tensorboard
%load_ext google.cloud.bigquery


下のセルの変数値を置き換えてください。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = PROJECT  # デフォルトは PROJECT
REGION = "us-central1"  # ご自身の REGION に置き換えてください

os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

In [ ]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

## BigQuery からのデータセット作成

Hacker News の見出しは、BigQuery の一般公開データセットとして入手可能です。[データセット](https://bigquery.cloud.google.com/table/bigquery-public-data:hacker_news.stories?tab=details)には、2006 年 10 月のサイト開設から 2015 年 10 月までのすべての見出しが含まれています。

データセットのサンプルは以下のとおりです。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    url, title, score
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    LENGTH(title) > 10
    AND score > 10
    AND LENGTH(url) > 0
LIMIT 10

BigQuery で正規表現の解析を行い、URL からニュース記事のソースを取得しましょう。例えば、URL が http://mobile.nytimes.com/.... の場合、<i>nytimes</i> だけを残したいと考えます。

In [ ]:
%%bigquery --project $PROJECT

SELECT
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.'))[SAFE_OFFSET(1)] AS source,
    COUNT(title) AS num_articles
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '.*://(.[^/]+)/'), '.com$')
    AND LENGTH(title) > 10
GROUP BY
    source
ORDER BY num_articles DESC
  LIMIT 100

URL を解析してソースを取得できるようになったので、ソースとタイトルのデータセットをまとめましょう。これが機械学習用のラベル付きデータセットになります。

In [ ]:
regex = ".*://(.[^/]+)/"


sub_query = """
SELECT
    title,
    ARRAY_REVERSE(SPLIT(REGEXP_EXTRACT(url, '{0}'), '.'))[SAFE_OFFSET(1)] AS source
    
FROM
    `bigquery-public-data.hacker_news.full`
WHERE
    REGEXP_CONTAINS(REGEXP_EXTRACT(url, '{0}'), '.com$')
    AND LENGTH(title) > 10
""".format(
    regex
)


query = """
SELECT 
    LOWER(REGEXP_REPLACE(title, '[^a-zA-Z0-9 $.-]', ' ')) AS title,
    source
FROM
  ({sub_query})
WHERE (source = 'github' OR source = 'nytimes' OR source = 'techcrunch')
""".format(
    sub_query=sub_query
)

print(query)

機械学習の訓練では通常、データセットを訓練データセットと評価データセットに分割する必要があります（また、評価データセットに基づいてモデルや特徴量の選択を行う場合は、おそらく独立したテストデータセットも必要です）。ただし、AutoML は独自にこれらの分割を作成する方法を判断するため、ここでは分割を行う必要はありません。

In [ ]:
bq = bigquery.Client(project=PROJECT)
title_dataset = bq.query(query).to_dataframe()
title_dataset.head()

テキスト分類用の AutoML では以下が要求されます。
* データセットが CSV 形式であること
* 最初の列が分類対象のテキスト、またはテキストへの GCS パスであること
* 最後の列がテキストのラベルであること

BigQuery から取得したデータセットは、これらの要件を満たしています。

In [ ]:
print(f"The full dataset contains {len(title_dataset)} titles")

3 つのラベルのそれぞれについて、ラベルの数がほぼ同じであることを確認しましょう。

In [ ]:
title_dataset.source.value_counts()

最後に、現在メモリ上にあるデータをディスクに保存します。

完全なデータセットを含む CSV ファイルと、開発用の 1,000 件の記事のみを含む別の CSV ファイルを作成します。

**注:** 完全なデータセットで AutoML を訓練するには長い時間がかかる場合があるため、ツールを学習する目的にはサンプルデータセットを使用することをお勧めします。

In [ ]:
DATADIR = "./data/"

if not os.path.exists(DATADIR):
    os.makedirs(DATADIR)

In [ ]:
FULL_DATASET_NAME = "titles_full.csv"
FULL_DATASET_PATH = os.path.join(DATADIR, FULL_DATASET_NAME)

# ディスクに書き込む前にデータをシャッフルします。
title_dataset = title_dataset.sample(n=len(title_dataset))

title_dataset.to_csv(
    FULL_DATASET_PATH, header=False, index=False, encoding="utf-8"
)

## 再利用可能な埋め込みモデル

In [ ]:
import datetime
import os
import shutil

import keras
import pandas as pd
import tensorflow as tf
from keras.callbacks import EarlyStopping, TensorBoard
from keras.layers import Dense, Input, Lambda, TextVectorization
from keras.models import Model
from keras.utils import to_categorical
from tensorflow_hub import KerasLayer

print(tf.__version__)

In [ ]:
%matplotlib inline

まず、訓練済みモデルに関する情報が保存される場所と、データセットが配置されている場所を指定することから始めましょう。

In [ ]:
MODEL_DIR = f"gs://{BUCKET}/text_models"

## データセットの読み込み

前のラボと同様に、データセットは記事のタイトルと、それらの記事がどのソース（GitHub、TechCrunch、または The New York Times）から取得されたかを示すラベルで構成されています。

In [ ]:
ls $DATADIR

In [ ]:
DATASET_NAME = "titles_full.csv"
TITLE_SAMPLE_PATH = os.path.join(DATADIR, DATASET_NAME)
COLUMNS = ["title", "source"]

titles_df = pd.read_csv(TITLE_SAMPLE_PATH, header=None, names=COLUMNS)
titles_df.head()

バランスの取れたデータセットであることを確認するために、ラベルごとのサンプル数を再度見てみましょう。

In [ ]:
titles_df.source.value_counts()

## ラベルの準備

このラボでは、モデルの最初のレイヤに英語用の事前学習済み [Kaggle Hub 埋め込みモジュール](https://tfhub.dev/s?q=tf2%20embeddings%20text%20english)を使用します。これを行う直接的なメリットは、Kaggle Hub の埋め込みモジュールが生テキストの処理を行ってくれる点です。
これは、モデルが単語を表す整数のシーケンスの代わりに、テキストを直接受け取ることができることも意味します。

ただし、前と同様に、ラベルをワンホットエンコードされたベクトルに前処理する必要があります。

In [ ]:
CLASSES = {"github": 0, "nytimes": 1, "techcrunch": 2}
N_CLASSES = len(CLASSES)

In [ ]:
def encode_labels(sources):
    classes = [CLASSES[source] for source in sources]
    one_hots = to_categorical(classes, num_classes=N_CLASSES)
    return one_hots

In [ ]:
encode_labels(titles_df.source[:4])

## 訓練/テスト分割の準備

データを訓練分割とテスト分割に分割しましょう。

In [ ]:
N_TRAIN = int(len(titles_df) * 0.95)

titles_train, sources_train = (
    titles_df.title[:N_TRAIN],
    titles_df.source[:N_TRAIN],
)

titles_valid, sources_valid = (
    titles_df.title[N_TRAIN:],
    titles_df.source[N_TRAIN:],
)

念のため、訓練分割とテスト分割でクラスあたりのサンプル数がほぼ同じであることを確認します。

均等になっているため、モデルのパフォーマンスを測定するための適切な指標として正解率（accuracy）を使用できます。

In [ ]:
sources_train.value_counts()

In [ ]:
sources_valid.value_counts()

それでは、モデルに入力する特徴量とラベルを作成しましょう。

In [ ]:
X_train, Y_train = titles_train.to_numpy(), encode_labels(sources_train)
X_valid, Y_valid = titles_valid.to_numpy(), encode_labels(sources_valid)

In [ ]:
X_train[:3]

In [ ]:
Y_train[:3]

## NNLM モデル

まず、[ニューラル確率的言語モデル（Neural Probabilistic Language Model: NNLM）](http://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) を使用して事前学習された単語埋め込みを試します。Kaggle Hub には [nnlm-en-dim50-with-normalization](https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1) と呼ばれる 50 次元のモデルがあり、生成されたベクトルも正規化されます。

URL から読み込まれた Kaggle Hub モジュールは、Keras モデルのレイヤとして使用できます。事前学習済み埋め込み自体のパラメータをファインチューニングするのに十分なデータがあるため、事前学習済み埋め込みを読み込む `KerasLayer` で `trainable=True` を設定します。

In [ ]:
# TODO 1
NNLM = "https://tfhub.dev/google/nnlm-en-dim50/2"

nnlm_module = KerasLayer(
    NNLM, output_shape=[50], input_shape=[], dtype=tf.string, trainable=True
)

この事前学習済みテキスト埋め込みモデルは、入力文ごとに 50 次元の密な特徴量ベクトルを生成します。出力の次元数は、事前学習済みモデルのアーキテクチャの固定された特性です。

In [ ]:
# TODO 1
nnlm_module(
    keras.ops.convert_to_tensor(
        ["The dog is happy to see people in the street."]
    )
)

## Swivel モデル

次に、単語の共起行列を本質的に因数分解して単語埋め込みを作成するアルゴリズムである [Swivel](https://arxiv.org/abs/1602.02215) を使用して得られた単語埋め込みを試します。
Kaggle Hub は、事前学習済みの 20 次元 Swivel モジュール [gnews-swivel-20dim-with-oov](https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim-with-oov/1) をホストしています。

In [ ]:
# TODO 1
SWIVEL = "https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim-with-oov/1"
swivel_module = KerasLayer(
    SWIVEL, output_shape=[20], input_shape=[], dtype=tf.string, trainable=True
)

前の事前学習済み埋め込みと同様に、文を渡すと単一のベクトルが出力されます。

In [ ]:
# TODO 1
swivel_module(
    keras.ops.convert_to_tensor(
        ["The dog is happy to see people in the street."]
    )
)

## モデルの構築

この関数 `build_model` は、事前学習済みの Kaggle Hub 埋め込みレイヤを活用して、テキスト分類用に設計された Keras モデルを構築します。訓練および評価が可能なニューラルネットワークアーキテクチャを動的に作成します。


構築するモデルの構成と仕組みの内訳は以下のとおりです。

- 入力層（Input Layer）: モデルは、未加工のテキスト文字列を受け入れるように構成された Input レイヤの定義から始まります。これにより、モデルはテキストを事前に数値形式に変換することなく直接処理できます。

- 埋め込み層（Embedding Layer）: 事前学習済みの TensorFlow Hub モデルである `hub_module` は、Lambda レイヤでラップされます。これは TensorFlow Hub モジュールを Keras Functional API と互換性を持たせ、他の任意の Keras レイヤと同様に使用できるようにするために必要な手順です。`module_output_dim` パラメータは、`hub_module` が生成する埋め込みの次元数を指定します。

- 分類ヘッド（Classification Head）: 埋め込み層に続いて、2 つ（またはそれ以上にする場合はそれ以上）の全結合（Dense）レイヤがネットワークの分類部分を形成します。

モデルの組み立てとコンパイル: Model は入力レイヤと出力レイヤを指定して作成されます。その後、adam オプティマイザ、損失関数として categorical_crossentropy（マルチクラス分類の標準）、評価指標として accuracy を指定してコンパイルされます。

In [ ]:
def build_model(hub_module, module_output_dim, name):
    # 生テキスト文字列用の入力層を定義
    inputs = Input(shape=(), dtype=tf.string, name="text_input")

    # KerasTensor を出力できるように hub_module の呼び出しを Lambda レイヤでラップ
    x = Lambda(
        lambda tensor: hub_module(tensor), output_shape=(module_output_dim,)
    )(inputs)

    # Dense レイヤを追加
    x = Dense(16, activation="relu")(x)

    # 分類用の最終 Dense レイヤを追加
    outputs = Dense(N_CLASSES, activation="softmax")(x)

    # モデルを作成
    model = Model(inputs=inputs, outputs=outputs, name=name)

    # モデルをコンパイル
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=False,
    )
    return model

訓練コードを以下の `train_and_evaluate` 関数にラップしましょう。
* 訓練データと検証データ、コンパイル済みモデル自体、および `batch_size` を入力として受け取る
* コンパイル済みモデルを最大 100 エポック訓練し、検証損失が減少しなくなったときに早期終了（early-stopping）を行う
* 学習曲線をプロットするのに役立つ `history` オブジェクトを返す

In [ ]:
def train_and_evaluate(train_data, val_data, model, batch_size=5000):
    X_train, Y_train = train_data

    model_dir = os.path.join(MODEL_DIR, model.name)
    if tf.io.gfile.exists(model_dir):
        tf.io.gfile.rmtree(model_dir)

    history = model.fit(
        X_train,
        Y_train,
        epochs=100,
        batch_size=batch_size,
        validation_data=val_data,
        callbacks=[EarlyStopping(patience=3), TensorBoard(model_dir)],
    )
    return history

## NNLM の訓練

In [ ]:
data = (X_train, Y_train)
val_data = (X_valid, Y_valid)

In [ ]:
nnlm_model = build_model(
    hub_module=nnlm_module, module_output_dim=50, name="nnlm"
)
nnlm_history = train_and_evaluate(data, val_data, nnlm_model)

In [ ]:
history = nnlm_history
pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

## Swivel の訓練

In [ ]:
swivel_model = build_model(
    hub_module=swivel_module, module_output_dim=20, name="swivel"
)

In [ ]:
swivel_history = train_and_evaluate(data, val_data, swivel_model)

In [ ]:
history = swivel_history
pd.DataFrame(history.history)[["loss", "val_loss"]].plot()
pd.DataFrame(history.history)[["accuracy", "val_accuracy"]].plot()

## モデルの比較

静的なプロットも役立ちますが、インタラクティブな可視化を使用すると、2 つのモデルのパフォーマンスをより直接的に比較できます。TensorBoard を使用すると、Swivel モデルと NNLM モデルの両方の指標を単一のインタラクティブなグラフ上に重ねて表示できます。これにより、Swivel の方が訓練が速いものの、NNLM の方がより少ない訓練エポックで優れた検証精度に収束することが明確にわかります。これらの結果を確認するには、次のコマンドで TensorBoard ダッシュボードを起動します。

In [ ]:
%tensorboard --logdir $MODEL_DIR --port 8008

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License